# Ajuste fino de un modelo de visión

**Sesión 4 · IA Intermedio · EstudIA 2026**

En la sesión 3, la mejor red convolucional entrenada desde cero clasificó bien unas 9 de cada 10 imágenes
satelitales. Hoy tomamos un modelo que **otra persona entrenó con más de un millón de fotos** y lo
**ajustamos** a esas mismas imágenes con el `Trainer` de Hugging Face.

**El recorrido.**

| Parte | Qué hacemos |
|---|---|
| 1 | Los datos: las imágenes satelitales de la sesión 3 |
| 2 | El modelo preentrenado, tal cual |
| 3 | Una cabeza nueva |
| 4 | El `Trainer` de Hugging Face |
| 5 | Ajuste 1: solo la cabeza |
| 6 | Ajuste 2: todo el modelo |
| 7 | Con pocos datos |
| 8 | Comparar |
| 9 | Dónde se equivoca |
| 10 | Tu modelo ajustado, listo para usar |

**Antes de empezar.**

1. *Archivo › Guardar una copia en Drive*.
2. Activa la GPU: *Entorno de ejecución › Cambiar tipo de entorno de ejecución › GPU T4 › Guardar*.
   Con GPU, cada ajuste tarda uno o dos minutos. Sin GPU tardaría demasiado: en ese caso el notebook
   carga los resultados de los ajustes que hice yo con GPU y sigues igual el recorrido.
3. Ejecuta la celda de preparación (▶ o `Shift + Enter`).

In [ ]:
# Preparación: librerías y GPU. Ejecútala una vez.
import os, json, time, urllib.request
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s04")

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from transformers import (AutoImageProcessor, AutoModelForImageClassification, Trainer, TrainingArguments,
                          pipeline)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_warning()

GPU = torch.cuda.is_available()

if GPU:
    print("Listo. GPU:", torch.cuda.get_device_name(0))
else:
    print("Listo, pero sin GPU. Actívala en Entorno de ejecución › Cambiar tipo de entorno de ejecución › GPU T4\n"
          "y vuelve a ejecutar esta celda. Si Colab no te da una, el notebook carga los resultados que obtuve con GPU.")

In [ ]:
# @title Funciones auxiliares: descargar archivos, escribir números y dibujar curvas
def descargar(nombre):
    """Baja un archivo de la carpeta pública del curso (solo la primera vez) y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def curvas(historia, titulo, metrica="exactitud"):
    """Pérdida de entrenamiento y de validación, y la métrica de validación, por época."""
    epocas = range(1, len(historia["val_loss"]) + 1)
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
    ejes[0].plot(epocas, historia["loss"], label="Entrenamiento", color="#000000", lw=2)
    ejes[0].plot(epocas, historia["val_loss"], label="Validación", color="#CB318E", lw=2)
    ejes[0].set_title("Pérdida"); ejes[0].legend(frameon=False, fontsize=9)
    ejes[1].plot(epocas, historia["val_metrica"], color="#CB318E", lw=2)
    ejes[1].set_title(f"{metrica.capitalize()} en validación"); ejes[1].set_ylim(0, 1)
    for eje in ejes:
        eje.set_xlabel("Época"); eje.grid(alpha=0.3); eje.spines[["top", "right"]].set_visible(False)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); plt.show()

def historia_de(trainer):
    """Saca de trainer.state.log_history la pérdida por época y lo que midió en validación."""
    h = {"loss": [], "val_loss": [], "val_metrica": []}
    for fila in trainer.state.log_history:
        if "loss" in fila and "eval_loss" not in fila:
            h["loss"].append(round(float(fila["loss"]), 5))
        if "eval_loss" in fila:
            h["val_loss"].append(round(float(fila["eval_loss"]), 5))
            h["val_metrica"].append(round(float(fila["eval_metrica"]), 5))
    return h

---
## 1. Los datos: las imágenes satelitales de la sesión 3

Las mismas 2.500 imágenes con etiqueta de la competencia de la sesión 3: cuadros de 32 × 32 píxeles
vistos por el satélite Sentinel-2, en 10 clases de uso del suelo. Para medir, usamos 2.000 imágenes
nuevas **con** su etiqueta (200 por clase), que la competencia no usó.

In [ ]:
datos = np.load(descargar("satelite_entrenamiento.npz"))
prueba = np.load(descargar("satelite_prueba.npz"))
CLASES = sorted(set(datos["etiquetas"].tolist()))
X = datos["imagenes"]
y = np.array([CLASES.index(e) for e in datos["etiquetas"]])
X_prueba = prueba["imagenes"]
y_prueba = np.array([CLASES.index(e) for e in prueba["etiquetas"]])

# 20 % para validación, con el mismo número de imágenes por clase
i_ent, i_val = train_test_split(np.arange(len(X)), test_size=0.2, stratify=y, random_state=42)
X_ent, y_ent, X_val, y_val = X[i_ent], y[i_ent], X[i_val], y[i_val]
print(f"Entrenamiento: {miles(len(X_ent))} · Validación: {miles(len(X_val))} · Prueba: {miles(len(X_prueba))}")

fig, ejes = plt.subplots(1, 10, figsize=(15, 2))
for c, eje in enumerate(ejes):
    eje.imshow(X_ent[np.where(y_ent == c)[0][0]]); eje.axis("off")
    eje.set_title(CLASES[c].replace("_", " "), fontsize=8)
plt.show()

---
## 2. El modelo preentrenado, tal cual

Usamos [`facebook/convnext-tiny-224`](https://huggingface.co/facebook/convnext-tiny-224) (licencia Apache 2.0):
una red convolucional moderna de unos 28 millones de parámetros, entrenada con ImageNet, más de un millón
de fotos en 1.000 clases. Veamos qué dice de nuestras imágenes, sin cambiarle nada.

In [ ]:
MODELO_BASE = "facebook/convnext-tiny-224"
imagenet = pipeline("image-classification", model=MODELO_BASE, device=0 if GPU else -1)

muestra = [int(np.where(y_val == c)[0][0]) for c in (1, 5, 6, 8)]
fig, ejes = plt.subplots(1, len(muestra), figsize=(13, 3.2))
for eje, i in zip(ejes, muestra):
    foto = Image.fromarray(X_val[i]).resize((224, 224), Image.BICUBIC)
    r = imagenet(foto, top_k=2)
    eje.imshow(foto); eje.axis("off")
    lineas = [f"{p['label'][:28]} {p['score']:.2f}".replace(".", ",") for p in r]
    eje.set_title(f"Es: {CLASES[y_val[i]]}\n" + "\n".join(lineas), fontsize=9)
plt.tight_layout(); plt.show()

**Para pensar.**

- ¿Por qué responde cosas como “dugongo”, “terciopelo” o “avión de pasajeros”?
- ¿Puede decir “bosque” o “río”? ¿Cuáles son las únicas respuestas que puede dar?

---
## 3. Una cabeza nueva

El modelo tiene dos partes: el **cuerpo**, que convierte la foto en un vector de 768 números, y la
**cabeza**, una capa que convierte ese vector en una probabilidad para cada una de las 1.000 clases de
ImageNet. Le pedimos a Hugging Face el mismo modelo con una cabeza de **10 salidas**, una por clase
nuestra. El cuerpo llega con lo que aprendió; la cabeza llega con pesos al azar.

Fíjate en el reporte que imprime Hugging Face: los pesos de la cabeza (`classifier.weight` y
`classifier.bias`) dicen `MISMATCH` y `Reinit`. No cabían en la cabeza nueva, así que empiezan al azar.
Todos los demás pesos llegaron del modelo preentrenado.

In [ ]:
modelo = AutoModelForImageClassification.from_pretrained(
    MODELO_BASE,
    id2label=dict(enumerate(CLASES)), label2id={c: i for i, c in enumerate(CLASES)},
    ignore_mismatched_sizes=True)   # la cabeza de 1.000 salidas no sirve: crea una nueva de 10

cuerpo = sum(p.numel() for p in modelo.convnext.parameters())
cabeza = sum(p.numel() for p in modelo.classifier.parameters())
print(f"Cuerpo: {miles(cuerpo)} parámetros, aprendidos con ImageNet.")
print(f"Cabeza: {miles(cabeza)} parámetros, al azar: 768 × 10 pesos + 10 sesgos.")

---
## 4. El `Trainer` de Hugging Face

En la sesión 3 escribimos el ciclo de entrenamiento a mano: predecir, medir el error, calcular
gradientes y ajustar. El `Trainer` hace ese ciclo por nosotros. Le damos cuatro cosas:

| Pieza | Qué es |
|---|---|
| `model` | El modelo, con su cabeza nueva |
| `args` | Un `TrainingArguments`: épocas, tasa de aprendizaje, tamaño del lote, cada cuánto validar |
| `train_dataset` y `eval_dataset` | Los datos, como una lista de ejemplos |
| `data_collator` | Cómo juntar ejemplos en un lote: aquí agrandamos las imágenes a 224 × 224 (el tamaño con el que aprendió el modelo), las normalizamos como en ImageNet y, solo en el entrenamiento, aplicamos aumento de datos |

La función `ajustar` arma todo eso. Con `congelar=True`, el cuerpo no se entrena: solo aprende la cabeza.

In [ ]:
PROCESADOR = AutoImageProcessor.from_pretrained(MODELO_BASE)
MEDIA = torch.tensor(PROCESADOR.image_mean).view(1, 3, 1, 1)
DESV = torch.tensor(PROCESADOR.image_std).view(1, 3, 1, 1)

class Imagenes(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return {"pixel_values": torch.from_numpy(self.X[i]), "labels": int(self.y[i])}

def juntar(lote, aumentar=False):
    """Junta ejemplos en un lote: agranda a 224 × 224, normaliza y, si aumentar=True, voltea y gira."""
    x = torch.stack([e["pixel_values"] for e in lote]).permute(0, 3, 1, 2).float() / 255
    if aumentar:   # una foto satelital no tiene “arriba”: voltear y girar 90° no cambian la etiqueta
        if torch.rand(1) < 0.5: x = x.flip(3)
        if torch.rand(1) < 0.5: x = x.flip(2)
        x = torch.rot90(x, int(torch.randint(4, (1,))), (2, 3))
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
    return {"pixel_values": (x - MEDIA) / DESV, "labels": torch.tensor([e["labels"] for e in lote])}

def metrica(prediccion):
    """La exactitud en validación, que el Trainer calcula al final de cada época."""
    return {"metrica": float((prediccion.predictions.argmax(-1) == prediccion.label_ids).mean())}

TIEMPOS = {}   # segundos de cada ajuste

def ajustar(nombre, X_a, y_a, congelar, epocas, tasa):
    """Ajusta el modelo preentrenado con X_a, y_a. Devuelve el Trainer, la historia y las predicciones de prueba."""
    if not GPU:
        return None, *cargar_resultado(nombre)
    torch.manual_seed(0)
    modelo = AutoModelForImageClassification.from_pretrained(
        MODELO_BASE, id2label=dict(enumerate(CLASES)),
        label2id={c: i for i, c in enumerate(CLASES)}, ignore_mismatched_sizes=True)
    if congelar:
        for p in modelo.convnext.parameters():
            p.requires_grad = False
    args = TrainingArguments(
        output_dir=f"ajustes/{nombre}", num_train_epochs=epocas, learning_rate=tasa,
        per_device_train_batch_size=32, per_device_eval_batch_size=128,
        warmup_steps=0.1,                 # la tasa sube durante el primer 10 % de los pasos
        eval_strategy="epoch", logging_strategy="epoch", save_strategy="no",
        fp16=True, report_to="none", seed=0, remove_unused_columns=False)
    trainer = Trainer(model=modelo, args=args,
                      train_dataset=Imagenes(X_a, y_a), eval_dataset=Imagenes(X_val, y_val),
                      data_collator=lambda lote: juntar(lote, aumentar=True), compute_metrics=metrica)
    inicio = time.time()
    trainer.train()
    segundos = TIEMPOS[nombre] = time.time() - inicio
    trainer.data_collator = juntar   # sin aumento para medir y predecir
    pred = trainer.predict(Imagenes(X_prueba, y_prueba)).predictions.argmax(-1)
    print(f"{nombre}: {epocas} épocas con {miles(len(X_a))} imágenes en {segundos:.0f} s.")
    return trainer, historia_de(trainer), pred

In [ ]:
# @title Sin GPU: cargar los resultados de los ajustes que hice con GPU
RESULTADOS_GPU = None

def cargar_resultado(nombre):
    """Sin GPU: la historia y las predicciones de prueba de un ajuste que hice con GPU."""
    global RESULTADOS_GPU
    if RESULTADOS_GPU is None:
        with open(descargar("resultados_demo.json"), encoding="utf-8") as f:
            RESULTADOS_GPU = json.load(f)
    r = RESULTADOS_GPU[nombre]
    print(f"{nombre}: sin GPU, cargué el ajuste que hice con GPU ({r['segundos']:.0f} s en una {RESULTADOS_GPU['gpu']}).")
    return r["historia"], np.array(r["pred_prueba"])

---
## 5. Ajuste 1: solo la cabeza

Congelamos el cuerpo y entrenamos solo la cabeza: 7.690 parámetros. Como la cabeza empieza al azar,
usamos una tasa de aprendizaje como la de la sesión 3, `1e-3`.

In [ ]:
trainer_cabeza, historia_cabeza, pred_cabeza = ajustar("cabeza", X_ent, y_ent, congelar=True, epocas=5, tasa=1e-3)
curvas(historia_cabeza, "Solo la cabeza (cuerpo congelado)")
print("Exactitud en prueba:", pct((pred_cabeza == y_prueba).mean()))

---
## 6. Ajuste 2: todo el modelo

Ahora entrenamos los 28 millones de parámetros, con una tasa de aprendizaje **diez veces menor**,
`1e-4`: queremos que el cuerpo se acomode a las imágenes satelitales sin borrar lo que aprendió con
ImageNet.

In [ ]:
trainer_todo, historia_todo, pred_todo = ajustar("todo", X_ent, y_ent, congelar=False, epocas=5, tasa=1e-4)
curvas(historia_todo, "Todo el modelo")
print("Exactitud en prueba:", pct((pred_todo == y_prueba).mean()))

**Para pensar.**

- ¿Por qué ajustar todo el modelo le gana a ajustar solo la cabeza en estas imágenes?
- ¿Qué pasaría con una tasa de aprendizaje grande, como 1e-2, sobre todo el modelo?

---
## 7. Con pocos datos

¿Y si solo tuviéramos **25 imágenes por clase**, 250 en total? Es más o menos lo que tendrías en un
proyecto propio después de etiquetar a mano una tarde. Repetimos los dos ajustes con 20 épocas: con
tan pocas imágenes, cada época es muy corta.

In [ ]:
rng = np.random.default_rng(0)
pocas = np.concatenate([rng.choice(np.where(y_ent == c)[0], 25, replace=False) for c in range(len(CLASES))])
_, historia_pocas_cabeza, pred_pocas_cabeza = ajustar("pocas_cabeza", X_ent[pocas], y_ent[pocas], congelar=True, epocas=20, tasa=1e-3)
_, historia_pocas_todo, pred_pocas_todo = ajustar("pocas_todo", X_ent[pocas], y_ent[pocas], congelar=False, epocas=20, tasa=1e-4)
print("Con 25 por clase · solo la cabeza:", pct((pred_pocas_cabeza == y_prueba).mean()),
      "· todo el modelo:", pct((pred_pocas_todo == y_prueba).mean()))

---
## 8. Comparar

Antes de la clase entrené la mejor red de la sesión 3 desde cero (dos convoluciones por bloque,
normalización por lotes, aumento de datos y 80 épocas), con las mismas imágenes de entrenamiento, y la
medí con estas mismas 2.000 de prueba.

In [ ]:
DESDE_CERO = {"todas": 0.9115, "pocas": 0.7885}   # medido antes de la clase
filas = [("CNN desde cero (sesión 3)", DESDE_CERO["todas"], DESDE_CERO["pocas"]),
         ("Preentrenado: solo la cabeza", (pred_cabeza == y_prueba).mean(), (pred_pocas_cabeza == y_prueba).mean()),
         ("Preentrenado: todo el modelo", (pred_todo == y_prueba).mean(), (pred_pocas_todo == y_prueba).mean())]
fig, eje = plt.subplots(figsize=(9, 3.2))
pos = np.arange(len(filas))
eje.barh(pos + 0.2, [f[1] for f in filas], 0.38, color="#000000", label="2.000 imágenes de entrenamiento")
eje.barh(pos - 0.2, [f[2] for f in filas], 0.38, color="#CB318E", label="25 por clase (250)")
for p, f in zip(pos, filas):
    eje.text(f[1] + 0.01, p + 0.2, pct(f[1]), va="center", fontsize=9)
    eje.text(f[2] + 0.01, p - 0.2, pct(f[2]), va="center", fontsize=9)
eje.set_yticks(pos, [f[0] for f in filas]); eje.set_xlim(0, 1.12); eje.invert_yaxis()
eje.set_xlabel("Exactitud con las 2.000 imágenes de prueba"); eje.legend(frameon=False, loc="lower right", fontsize=9)
eje.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

**Para pensar.**

- ¿Cuánto pierde cada modelo cuando pasamos de 2.000 a 250 imágenes?
- Si tu proyecto tiene pocas imágenes etiquetadas, ¿qué camino elegirías?

---
## 9. Dónde se equivoca

La exactitud es un solo número. La **matriz de confusión** muestra qué confunde: cada fila es la clase
real y cada columna lo que respondió el modelo. Lo ideal es todo en la diagonal.

In [ ]:
fig, eje = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(confusion_matrix(y_prueba, pred_todo), display_labels=[c.replace("_", " ") for c in CLASES]).plot(
    ax=eje, cmap="Purples", colorbar=False, xticks_rotation=60)
eje.set_title("Todo el modelo, con las 2.000 de prueba"); eje.set_xlabel("Lo que respondió"); eje.set_ylabel("La clase real")
plt.tight_layout(); plt.show()
for c in range(len(CLASES)):
    print(f"{CLASES[c]:22s} recall {pct((pred_todo[y_prueba == c] == c).mean())}")

**Para pensar.**

- ¿Qué pares de clases confunde más? ¿Te parecen difíciles también para una persona?

---
## 10. Tu modelo ajustado, listo para usar

El modelo ajustado es un modelo de Hugging Face como cualquier otro: lo guardamos en una carpeta y lo
cargamos con `pipeline`, igual que en la sesión 3. Ahora responde con nuestras clases.

In [ ]:
if trainer_todo is None:
    print("Sin GPU no entrenamos aquí, así que no hay modelo que guardar. Con GPU, esta celda lo guarda y lo usa.")
else:
    trainer_todo.save_model("satelite-convnext")
    # crop_pct=1.0: el procesador agranda la imagen a 224 × 224 sin recortar los bordes, como hicimos al ajustar
    AutoImageProcessor.from_pretrained(MODELO_BASE, crop_pct=1.0).save_pretrained("satelite-convnext")
    mi_modelo = pipeline("image-classification", model="satelite-convnext", device=0)
    fig, ejes = plt.subplots(1, 5, figsize=(13, 3))
    for eje, i in zip(ejes, [3, 400, 800, 1200, 1600]):
        r = mi_modelo(Image.fromarray(X_prueba[i]), top_k=1)[0]
        eje.imshow(X_prueba[i]); eje.axis("off")
        puntaje = f"{r['score']:.2f}".replace(".", ",")
        eje.set_title(f"{r['label']} {puntaje}\n(es: {CLASES[y_prueba[i]]})", fontsize=9)
    plt.tight_layout(); plt.show()

**Fuente de los datos:** Helber, P., Bischke, B., Dengel, A. y Borth, D. (2019). EuroSAT: A Novel Dataset and
Deep Learning Benchmark for Land Use and Land Cover Classification. *IEEE Journal of Selected Topics in Applied
Earth Observations and Remote Sensing*. Licencia MIT. Imágenes de Copernicus Sentinel-2 (ESA). Copia en Hugging
Face: `timm/eurosat-rgb`. Reducimos las imágenes de 64 × 64 a 32 × 32 píxeles y renombramos las clases al
español. Modelo: Liu y otros (2022), ConvNeXt, `facebook/convnext-tiny-224`, licencia Apache 2.0.